In [50]:
import pandas as pd
import numpy as np

In [51]:
df = pd.read_csv('../data/curated/visualisation_df.csv')
df

,Unnamed: 0,id,region,regulated_dam,primary_purpose,primary_type,height,length,volume,surface,...,assessment,probability_of_failure,dam_repair_loss,damage_loss,business_interruption_loss,age,modification_count,years_from_modification,years_from_inspection,years_from_assessment
0,0,SOAD00072,Navaldia,Yes,Recreation,Earth,NaN,NaN,NaN,0.02364,...,Satisfactory,0.1258,20.8,296.9,8.1,NaN,0,NaN,10.0,NaN
1,1,SOAD00380,Navaldia,No,NaN,Earth,2.713,NaN,NaN,NaN,...,Not Available,0.0757,930.5,727.5,NaN,98.0,0,98.0,7.0,98.0
2,2,SOAD00610,Navaldia,Yes,Recreation,Earth,NaN,NaN,NaN,NaN,...,Satisfactory,0.1375,355.5,427.3,8.6,NaN,0,NaN,NaN,NaN
3,3,SOAD00862,Navaldia,Yes,NaN,NaN,NaN,NaN,NaN,NaN,...,Not Rated,0.1403,295.1,25.3,NaN,NaN,0,NaN,NaN,NaN
4,4,SOAD02091,Lyndrassia,Yes,Recreation,Earth,13.921,0.200,NaN,0.04728,...,Not Rated,0.0998,11.5,203.0,5.6,44.0,0,44.0,4.0,44.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20801,20801,SOAD16536,Lyndrassia,No,Flood Risk Reduction,Rockfill,35.872,1.667,7411000.0,480.62681,...,Not Available,0.0837,820.9,510.6,NaN,47.0,0,47.0,2.0,47.0
20802,20802,SOAD13145,Lyndrassia,No,Hydroelectric,Gravity,141.296,0.963,375000.0,405.40236,...,Not Available,0.0795,850.2,176.6,62.9,52.0,0,52.0,5.0,52.0
20803,20803,SOAD12688,Navaldia,No,Flood Risk Reduction,Earth,37.905,6.213,7370000.0,177.79053,...,Not Available,0.0972,922.4,490.6,NaN,71.0,0,71.0,3.0,71.0
20804,20804,SOAD02340,Navaldia,No,Flood Risk Reduction,Earth,46.431,4.133,6000000.0,998.93972,...,Not Available,0.0672,652.4,603.7,NaN,60.0,0,60.0,4.0,60.0


In [52]:
df['damage_loss'] = df['damage_loss'].fillna(0)
df['dam_repair_loss'] = df['dam_repair_loss'].fillna(0)
df['business_interruption_loss'] = df['business_interruption_loss'].fillna(0)

In [53]:
df['business_interruption_loss'].isna().sum()

0

In [54]:
df['total_loss'] = df['damage_loss'] + df['dam_repair_loss'] + df['business_interruption_loss']
df['expected_loss'] = df['probability_of_failure'] * (df['damage_loss'] + df['dam_repair_loss'] + df['business_interruption_loss'])

### hazard

In [55]:
def calculate_hazard_rating_factor(df, hazard):
    curr_hazard_mean = (df[df['hazard'] == hazard].describe()['total_loss'].loc['50%'] + df[df['hazard'] == hazard].describe()['total_loss'].loc['50%']) / 2
    low_hazard_mean = (df[df['hazard'] == 'Low'].describe()['total_loss'].loc['50%'] + df[df['hazard'] == 'Low'].describe()['total_loss'].loc['50%']) / 2
    hazard_rating_factor = curr_hazard_mean/low_hazard_mean
    return hazard_rating_factor

In [56]:
hazard_low_rf = calculate_hazard_rating_factor(df, 'Low')
hazard_high_rf = calculate_hazard_rating_factor(df, 'High')
hazard_significant_rf = calculate_hazard_rating_factor(df, 'Significant')
hazard_undetermined_rf = calculate_hazard_rating_factor(df, 'Undetermined')

In [57]:
# Define mapping dictionary
hazard_mapping = {
    'Low': hazard_low_rf,
    'High': hazard_high_rf,
    'Significant': hazard_significant_rf,
    'Undetermined': hazard_undetermined_rf
}

# Create a new column using map()
df['hazard_rating_factor'] = df['hazard'].map(hazard_mapping)


In [58]:
df['region'].value_counts()

Navaldia      8878
Lyndrassia    8406
Flumevale     3522
Name: region, dtype: int64

### Regulation

In [59]:
df['no_BI_loss'] = df['dam_repair_loss'] + df['damage_loss']

In [60]:
df['w'] = df['no_BI_loss'] / df['total_loss']
df['w'] = df['w'].fillna(1)
df['failure_rate'] = df['probability_of_failure'] * df['w']
def calculate_regulation_rating_factor(df, region):
    regulated_region_failure_rate = sum(df[(df['region'] == region) & (df['regulated_dam'] == 'Yes')]['failure_rate'])
    unregulated_region_failure_rate = sum(df[(df['region'] == region) & (df['regulated_dam'] == 'No')]['failure_rate'])
    regulated_rating_factor = unregulated_region_failure_rate / regulated_region_failure_rate 
    return regulated_rating_factor

In [61]:
# Define mapping dictionary
regulated_mapping = {
    'Navaldia': calculate_regulation_rating_factor(df, 'Navaldia'),
    'Lyndrassia': calculate_regulation_rating_factor(df, 'Lyndrassia'),
    'Flumevale': calculate_regulation_rating_factor(df, 'Flumevale')
}

# Create a new column using map()
df['regulated_rating_factor'] = df['region'].map(regulated_mapping)


In [62]:
df['region'].value_counts()

Navaldia      8878
Lyndrassia    8406
Flumevale     3522
Name: region, dtype: int64

### GDP

In [63]:
gdp_df = pd.read_csv('../data/raw/soaGDP.csv')
pop_df = pd.read_csv('../data/raw/soaPOP.csv')

In [64]:
pop_df

,Year,Flumevale,Lyndrassia,Navaldia,Tarrodan
0,2019,"45,363,514","7,067,855","39,808,697","92,240,066"
1,2020,"45,502,051","7,097,789","40,175,188","92,775,028"
2,2021,"45,651,175","7,131,024","40,565,887","93,348,086"
3,2022,"45,599,000","7,157,446","40,953,108","93,709,554"
4,2023,"45,311,937","7,239,138","42,148,205","94,699,280"


In [65]:
gdp_df

,Year,Flumevale,Lyndrassia,Navaldia,Tarrodan
0,2019,"3,306,924","369,632","2,544,348","6,220,904"
1,2020,"3,313,625","370,374","2,460,436","6,144,435"
2,2021,"3,691,682","407,195","2,798,390","6,897,267"
3,2022,"3,963,845","447,382","3,202,171","7,613,398"
4,2023,"4,197,489","480,201","3,396,363","8,074,053"


In [66]:
for feature in gdp_df.columns:
    gdp_df[feature] = gdp_df[feature].replace(to_replace=',', value= '', regex=True).astype(int)
    pop_df[feature] = pop_df[feature].replace(to_replace=',', value= '', regex=True).astype(int)

In [67]:
gdp_pp_df = pd.DataFrame({
    'Year': gdp_df['Year']
})
for feature in gdp_df.drop(columns=['Year']).columns:
    gdp_pp_df[feature] = gdp_df[feature]/pop_df[feature]

In [68]:
gdp_pp_df

,Year,Flumevale,Lyndrassia,Navaldia,Tarrodan
0,2019,0.072898,0.052298,0.063914,0.067443
1,2020,0.072824,0.052182,0.061243,0.066229
2,2021,0.080867,0.057102,0.068984,0.073888
3,2022,0.086928,0.062506,0.078191,0.081245
4,2023,0.092635,0.066334,0.080581,0.085260


In [69]:
for feature in gdp_pp_df.drop(columns=['Year', 'Tarrodan']):
    gdp_pp_df[feature] = gdp_pp_df[feature]/gdp_pp_df['Tarrodan']

In [71]:
regulated_mapping = {
    'Navaldia': gdp_pp_df.iloc[4]['Navaldia'],
    'Lyndrassia': gdp_pp_df.iloc[4]['Lyndrassia'],
    'Flumevale': gdp_pp_df.iloc[4]['Flumevale']
}

# Create a new column using map()
df['gdp_rating_factor'] = df['region'].map(regulated_mapping)


### summary

In [75]:
df['total_rating_factor'] = df['hazard_rating_factor'] * df['regulated_rating_factor'] * df['gdp_rating_factor']

In [76]:
df[['hazard_rating_factor', 'regulated_rating_factor', 'gdp_rating_factor', 'total_rating_factor']].describe()

,hazard_rating_factor,regulated_rating_factor,gdp_rating_factor,total_rating_factor
count,20806.000000,20806.000000,20806.000000,20806.000000
mean,2.811650,0.877295,0.901545,1.858818
std,2.627682,0.564648,0.112990,2.246975
min,0.084730,0.090300,0.778021,0.045576
25%,1.000000,0.569129,0.778021,0.537899
50%,1.000000,0.569129,0.945127,1.192321
75%,7.036313,1.532505,0.945127,2.086738
max,7.036313,1.532505,1.086506,8.389541
